# Day 43 — Concurrency and parallelism: the GIL, races, deadlocks, asyncio, work-depth and map-reduce

Two CPU-heavy jobs on two Python threads take as long as running them one after the
other; on two *processes* they take about half. Twenty 50 ms waits on twenty threads take
50 ms instead of a second. Which tool wins is decided by one question - is the job
computing or waiting? - and by how much of it can run at the same time at all
(work and depth).

The other half of the day: concurrency bugs, like yesterday's randomness bugs, do not
crash. A lost update, a deadlock that needs one ordering in twenty million, a mapper that
puts a word in the wrong bucket - so this notebook builds a small checker that runs
*every* interleaving of a few threads, the same idea as Day 42's "replay every random
choice".

Functions that a `multiprocessing.Pool` sends to worker processes are imported from
`concurrency.py` (next to this notebook): workers must be able to import them by name,
and a function defined in a notebook cell cannot be found by a freshly started process.

In [1]:
import asyncio
import hashlib
import math
import multiprocessing as mp
import os
import random
import sys
import threading
import time
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from functools import lru_cache, reduce

import concurrency            # the same code as a module, for the process pools

## 1. The GIL: threads overlap waiting, not computing

CPython runs Python bytecode on one thread at a time; the global interpreter lock (GIL)
is handed between threads every few milliseconds. A thread that sleeps or waits for the
network releases it, so waits overlap - computation does not. Separate processes each
have their own interpreter and their own GIL.

In [2]:
# ---------------------------------------------------------------- 1. the GIL
def burn(n: int) -> int:
    """Pure-Python CPU work: holds the GIL the whole time."""
    s = 0
    for i in range(n):
        s += i * i
    return s


def best_of(fn, repeat=3) -> float:
    best = float('inf')
    for _ in range(repeat):
        t = time.perf_counter()
        fn()
        best = min(best, time.perf_counter() - t)
    return best


def run_threads(target, args_list):
    ts = [threading.Thread(target=target, args=a) for a in args_list]
    for t in ts:
        t.start()
    for t in ts:
        t.join()


n = 2_000_000
serial = best_of(lambda: [burn(n) for _ in range(2)])
threads = best_of(lambda: run_threads(burn, [(n,), (n,)]))
with mp.Pool(2) as pool:
    pool.map(concurrency.burn, [10, 10])                     # start the workers
    procs = best_of(lambda: pool.map(concurrency.burn, [n, n]))
print(f'{os.cpu_count()} cores; 2 CPU jobs: serial {serial:.3f} s, threads {threads:.3f} s, '
      f'processes {procs:.3f} s')
t = best_of(lambda: [time.sleep(0.05) for _ in range(20)], 1)
u = best_of(lambda: run_threads(time.sleep, [(0.05,)] * 20), 1)
print(f'20 waits of 50 ms: serial {t:.3f} s, 20 threads {u:.3f} s')

2 cores; 2 CPU jobs: serial 0.367 s, threads 0.376 s, processes 0.233 s
20 waits of 50 ms: serial 1.005 s, 20 threads 0.057 s


## 2. A race you can run

`x = x + 1` is three steps: read, add, write. If another thread writes between my read
and my write, its update is lost. In this CPython the switch only happens at loop jumps
and function calls, so a bare `+=` happens not to be split - put one function call
between the read and the write and the updates start disappearing.

In [3]:
# ---------------------------------------------------------------- 2. races, for real
class Box:
    def __init__(self):
        self.v = 0


def _noop():
    pass


def bump_inline(box):
    box.v += 1                      # read, add, write - one line


def bump_with_call(box):
    v = box.v                       # read
    _noop()                         # any function call: logging, validation, ...
    box.v = v + 1                   # write the value read *before* the call


def hammer(bump, threads=4, each=200_000, switch=None, lock=None):
    """`threads` threads each call bump(box) `each` times; return the final count."""
    old = sys.getswitchinterval()
    if switch is not None:
        sys.setswitchinterval(switch)
    box = Box()

    def work():
        for _ in range(each):
            if lock is None:
                bump(box)
            else:
                with lock:
                    bump(box)
    try:
        run_threads(work, [()] * threads)
    finally:
        sys.setswitchinterval(old)
    return box.v


print('right answer 800,000')
print('box.v += 1               ', hammer(bump_inline))
print('read, call, write        ', hammer(bump_with_call))
print('same, under a lock       ', hammer(bump_with_call, lock=threading.Lock()))

right answer 800,000
box.v += 1                800000
read, call, write         316803
same, under a lock        800000


## 3. Every interleaving, counted

A run of real threads shows one interleaving out of many, chosen by the scheduler. The
checker below models each thread as a list of atomic steps and runs **every** order in
which they can interleave (each thread's own steps stay in order). Memoising on the state
lets it count billions of interleavings.

In [4]:
# ---------------------------------------------------------------- 3. an interleaving checker
# A tiny model of threads: each thread is a list of atomic steps.  The checker runs
# EVERY order in which the steps of different threads can interleave (each thread's
# own steps stay in order) - the same idea as Day 42's "replay every random choice".
#   ('read',  var, reg)        reg <- shared[var]
#   ('write', var, reg, add)   shared[var] <- reg + add
#   ('acquire', lock)          only possible while nobody holds the lock
#   ('release', lock)
#   ('wait', event)            only possible once the event is set
#   ('set', event)
#   ('emit', text)             append text to the output
DEADLOCK = 'DEADLOCK'


def _start(threads, shared):
    return (tuple(0 for _ in threads), tuple(sorted(shared.items())),
            tuple(() for _ in threads), (), frozenset(), '')


def _enabled(threads, state, i):
    pcs, shared, regs, held, events, out = state
    if pcs[i] >= len(threads[i]):
        return False
    op = threads[i][pcs[i]]
    if op[0] == 'acquire':
        return all(l != op[1] for l, _ in held)
    if op[0] == 'wait':
        return op[1] in events
    return True


def _step(threads, state, i):
    pcs, shared, regs, held, events, out = state
    op = threads[i][pcs[i]]
    sh, rg = dict(shared), dict(regs[i])
    held, events = list(held), set(events)
    kind = op[0]
    if kind == 'read':
        rg[op[2]] = sh[op[1]]
    elif kind == 'write':
        sh[op[1]] = rg[op[2]] + op[3]
    elif kind == 'acquire':
        held.append((op[1], i))
    elif kind == 'release':
        held.remove((op[1], i))
    elif kind == 'set':
        events.add(op[1])
    elif kind == 'emit':
        out += op[1]
    pcs = pcs[:i] + (pcs[i] + 1,) + pcs[i + 1:]
    regs = regs[:i] + (tuple(sorted(rg.items())),) + regs[i + 1:]
    return (pcs, tuple(sorted(sh.items())), regs, tuple(sorted(held)), frozenset(events), out)


def _outcome(threads, state):
    pcs, shared, regs, held, events, out = state
    if all(pcs[i] == len(t) for i, t in enumerate(threads)):
        return (shared, out)
    return DEADLOCK


def every_schedule(threads, shared):
    """Yield (schedule, outcome) for every interleaving.  Small cases only."""
    def go(state, sched):
        nxt = [i for i in range(len(threads)) if _enabled(threads, state, i)]
        if not nxt:
            yield tuple(sched), _outcome(threads, state)
            return
        for i in nxt:
            sched.append(i)
            yield from go(_step(threads, state, i), sched)
            sched.pop()
    yield from go(_start(threads, shared), [])


def count_schedules(threads, shared):
    """Counter outcome -> number of interleavings reaching it (memoised on the state,
    so it copes with billions of interleavings)."""
    @lru_cache(maxsize=None)
    def go(state):
        nxt = [i for i in range(len(threads)) if _enabled(threads, state, i)]
        if not nxt:
            return ((_outcome(threads, state), 1),)
        total = Counter()
        for i in nxt:
            for k, v in go(_step(threads, state, i)):
                total[k] += v
        return tuple(total.items())
    return Counter(dict(go(_start(threads, shared))))


def reachable_states(threads, shared):
    """(number of distinct states, number of them that are deadlocks)."""
    seen, stack, dead = set(), [_start(threads, shared)], 0
    while stack:
        s = stack.pop()
        if s in seen:
            continue
        seen.add(s)
        nxt = [i for i in range(len(threads)) if _enabled(threads, s, i)]
        if not nxt and _outcome(threads, s) == DEADLOCK:
            dead += 1
        stack.extend(_step(threads, s, i) for i in nxt)
    return len(seen), dead


def incrementer(locked=False, times=1):
    body = [('read', 'x', 'r'), ('write', 'x', 'r', 1)]
    if locked:
        body = [('acquire', 'L')] + body + [('release', 'L')]
    return body * times


def final_x(outcome):
    return DEADLOCK if outcome == DEADLOCK else dict(outcome[0])['x']


def xcount(counter):
    """Counter keyed by the final x (or DEADLOCK)."""
    out = Counter()
    for k, v in counter.items():
        out[final_x(k)] += v
    return out


for sched, out in every_schedule([incrementer(), incrementer()], {'x': 0}):
    print(' '.join('AB'[i] for i in sched), ' x =', final_x(out))
for k in (2, 3, 4):
    c = xcount(count_schedules([incrementer()] * k, {'x': 0}))
    print(f'{k} threads: {sum(c.values())} interleavings, {c[k]} correct', dict(sorted(c.items())))
print('with a lock:', xcount(count_schedules([incrementer(locked=True)] * 3, {'x': 0})))

A A B B  x = 2
A B A B  x = 1
A B B A  x = 1
B A A B  x = 1
B A B A  x = 1
B B A A  x = 2
2 threads: 6 interleavings, 2 correct {1: 4, 2: 2}
3 threads: 90 interleavings, 6 correct {1: 48, 2: 36, 3: 6}
4 threads: 2520 interleavings, 24 correct {1: 1152, 2: 1056, 3: 288, 4: 24}
with a lock: Counter({3: 6})


## 4. Deadlock, and LeetCode 1226 (Dining Philosophers)

Two threads that take two locks in opposite orders can each hold one and wait forever for
the other. Five philosophers who all pick up their left fork first can do the same in a
circle - but only in one state out of 9,302, which is why a test run almost never sees it.
Taking the lower-numbered fork first breaks the circle.

In [5]:
# ---------------------------------------------------------------- 4. deadlock
def two_lock_threads(same_order: bool):
    t1 = [('acquire', 'A'), ('acquire', 'B'), ('release', 'B'), ('release', 'A')]
    first, second = ('A', 'B') if same_order else ('B', 'A')
    t2 = [('acquire', first), ('acquire', second), ('release', second), ('release', first)]
    return [t1, t2]


def philosophers(n=5, ordered=False):
    """Philosopher i needs fork i (left) and fork (i+1) % n (right)."""
    ts = []
    for i in range(n):
        a, b = i, (i + 1) % n
        if ordered and a > b:
            a, b = b, a                  # always pick up the lower-numbered fork first
        ts.append([('acquire', f'f{a}'), ('acquire', f'f{b}'), ('emit', str(i)),
                   ('release', f'f{b}'), ('release', f'f{a}')])
    return ts


def real_deadlock(n=5, nap=0.05, patience=0.5, ordered=False):
    """Real threads and locks.
    Left fork first: everybody picks up a fork, waits until all have one (a Barrier),
    then looks at the fork they need next.  Returns who found it taken - that moment
    is the deadlock, caught without hanging the program.
    Lower-numbered fork first: everybody picks a fork, naps, then reaches for the
    other one with a timeout.  Returns who had to give up."""
    forks = [threading.Lock() for _ in range(n)]
    stuck = []
    all_hold_one = threading.Barrier(n)

    def phil(i):
        a, b = i, (i + 1) % n
        if ordered and a > b:
            a, b = b, a
        with forks[a]:
            if not ordered:
                all_hold_one.wait()
                if forks[b].locked():
                    stuck.append(i)
                all_hold_one.wait()           # nobody lets go before everybody looked
                return
            time.sleep(nap)
            if forks[b].acquire(timeout=patience):
                forks[b].release()
            else:
                stuck.append(i)
    run_threads(phil, [(i,) for i in range(n)])
    return sorted(stuck)


class DiningPhilosophers:
    """LeetCode 1226: forks are taken lowest-number first, so no cycle can form."""

    def __init__(self, n=5):
        self.n = n
        self.forks = [threading.Lock() for _ in range(n)]

    def wantsToEat(self, philosopher, pickLeftFork, pickRightFork, eat,
                   putLeftFork, putRightFork):
        left, right = philosopher, (philosopher + 1) % self.n
        lo, hi = min(left, right), max(left, right)
        with self.forks[lo], self.forks[hi]:
            pickLeftFork()
            pickRightFork()
            eat()
            putLeftFork()
            putRightFork()


def lc1226_run(n=5, meals=3):
    """Every philosopher eats `meals` times; returns (meals eaten, action log)."""
    table, log = DiningPhilosophers(n), []

    def phil(i):
        left, right = i, (i + 1) % n
        for _ in range(meals):
            table.wantsToEat(i, lambda: log.append((i, 'pick', left)),
                             lambda: log.append((i, 'pick', right)),
                             lambda: log.append((i, 'eat', None)),
                             lambda: log.append((i, 'put', left)),
                             lambda: log.append((i, 'put', right)))
    run_threads(phil, [(i,) for i in range(n)])
    return sum(1 for _, a, _ in log if a == 'eat'), log


def forks_never_shared(log) -> bool:
    holder = {}
    for who, action, fork in log:
        if action == 'pick':
            if fork in holder:
                return False
            holder[fork] = who
        elif action == 'put':
            if holder.pop(fork, None) != who:
                return False
    return not holder


for same in (False, True):
    print('two locks, same order' if same else 'two locks, opposite order',
          count_schedules(two_lock_threads(same), {}))
for n in (2, 3, 4, 5):
    c = count_schedules(philosophers(n), {})
    print(f'{n} philosophers: states, deadlocked = {reachable_states(philosophers(n), {})};'
          f' {c[DEADLOCK]:,} of {sum(c.values()):,} interleavings deadlock')
print('lower fork first:', reachable_states(philosophers(5, ordered=True), {}))
print('real threads, left first, who finds the other fork taken:', real_deadlock())
print('real threads, lower first, who had to give up:', real_deadlock(ordered=True))
meals, log = lc1226_run()
print('LC 1226:', meals, 'meals, forks never shared:', forks_never_shared(log))

two locks, opposite order Counter({((), ''): 4, 'DEADLOCK': 2})
two locks, same order Counter({((), ''): 2})
2 philosophers: states, deadlocked = (24, 1); 2 of 6 interleavings deadlock
3 philosophers: states, deadlocked = (140, 1); 6 of 234 interleavings deadlock
4 philosophers: states, deadlocked = (1050, 1); 24 of 1,167,000 interleavings deadlock
5 philosophers: states, deadlocked = (9302, 1); 120 of 2,430,842,120 interleavings deadlock
lower fork first: (8447, 0)
real threads, left first, who finds the other fork taken: [0, 1, 2, 3, 4]
real threads, lower first, who had to give up: []
LC 1226: 15 meals, forks never shared: True


## 5. LeetCode 1114 - Print in Order

Three threads call `first`, `second` and `third` in any order; the output must always be
"firstsecondthird". Two `threading.Event`s make `second` wait for `first` and `third` wait
for `second`.

In [6]:
# ---------------------------------------------------------------- 5. LeetCode 1114
def print_in_order_model(synced: bool):
    names = ['first', 'second', 'third']
    if not synced:
        return [[('emit', w[0])] for w in names]
    return [[('emit', 'f'), ('set', 'e1')],
            [('wait', 'e1'), ('emit', 's'), ('set', 'e2')],
            [('wait', 'e2'), ('emit', 't')]]


class Foo:
    """LeetCode 1114: second() waits for first(), third() waits for second()."""

    def __init__(self):
        self.done1 = threading.Event()
        self.done2 = threading.Event()

    def first(self, printFirst):
        printFirst()
        self.done1.set()

    def second(self, printSecond):
        self.done1.wait()
        printSecond()
        self.done2.set()

    def third(self, printThird):
        self.done2.wait()
        printThird()


def lc1114_all_orders():
    from itertools import permutations
    outs = []
    for order in permutations(range(3)):
        foo, out = Foo(), []
        calls = [lambda: foo.first(lambda: out.append('first')),
                 lambda: foo.second(lambda: out.append('second')),
                 lambda: foo.third(lambda: out.append('third'))]
        ts = [threading.Thread(target=calls[k]) for k in order]
        for t in ts:
            t.start()
            time.sleep(0.005)            # give the early starters a head start
        for t in ts:
            t.join()
        outs.append(''.join(out))
    return outs


print('no sync    :', sorted(o[1] for o in count_schedules(print_in_order_model(False), {})))
print('with events:', sorted(o[1] for o in count_schedules(print_in_order_model(True), {})))
print('real threads, every start order:', set(lc1114_all_orders()))

no sync    : ['fst', 'fts', 'sft', 'stf', 'tfs', 'tsf']
with events: ['fst']
real threads, every start order: {'firstsecondthird'}


## 6. asyncio: one thread, many waits

`await` hands control back to the event loop while a task waits, so ten 0.1 s waits take
0.1 s. A blocking call such as `time.sleep` inside an `async def` never hands control
back: the ten tasks run one after another and nothing reports an error.
`asyncio.Semaphore(3)` caps how many run at once.

(Jupyter already runs an event loop, so `run_coro` starts the coroutine on a helper thread
there; in a script it is just `asyncio.run`.)

In [7]:
# ---------------------------------------------------------------- 6. asyncio
async def fetch_await(i, delay, log, t0):
    log.append((i, 'start', time.perf_counter() - t0))
    await asyncio.sleep(delay)          # hands control back to the event loop
    log.append((i, 'end', time.perf_counter() - t0))
    return i


async def fetch_blocking(i, delay, log, t0):
    log.append((i, 'start', time.perf_counter() - t0))
    time.sleep(delay)                   # blocks the only thread the loop has
    log.append((i, 'end', time.perf_counter() - t0))
    return i


def run_coro(coro):
    """asyncio.run(coro) - also from inside Jupyter, which already runs an event loop
    (asyncio.run refuses to start a second loop in the same thread)."""
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)
    with ThreadPoolExecutor(1) as ex:
        return ex.submit(asyncio.run, coro).result()


def run_async(fetch, n=10, delay=0.1, limit=None):
    """Run n fetches with asyncio.gather (at most `limit` at a time).
    Returns (seconds, results, log)."""
    log = []

    async def main():
        t0 = time.perf_counter()
        sem = asyncio.Semaphore(limit or n)

        async def one(i):
            async with sem:
                return await fetch(i, delay, log, t0)
        res = await asyncio.gather(*(one(i) for i in range(n)))
        return time.perf_counter() - t0, res
    secs, res = run_coro(main())
    return secs, res, log


for name, fetch, limit in (('await asyncio.sleep', fetch_await, None),
                           ('time.sleep in async def', fetch_blocking, None),
                           ('Semaphore(3)', fetch_await, 3)):
    secs, res, log = run_async(fetch, limit=limit)
    print(f'{name:<24} {secs:.2f} s  starts', [round(t, 1) for i, k, t in log if k == 'start'])

await asyncio.sleep      0.10 s  starts [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
time.sleep in async def  1.00 s  starts [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
Semaphore(3)             0.40 s  starts [0.0, 0.0, 0.0, 0.1, 0.1, 0.1, 0.2, 0.2, 0.2, 0.3]


## 7. LeetCode 1242 - Web Crawler Multithreaded

Crawl every page reachable from a start URL that has the same host name, where each
`getUrls` call is a slow network fetch. Fetching a whole breadth-first wave at once makes
the time "number of waves x one fetch" instead of "number of pages x one fetch" - the
depth, not the work. Comparing host names with `in` instead of `==` lets
`news.yahoo.com.evil.example` in.

In [8]:
# ---------------------------------------------------------------- 7. LeetCode 1242
def hostname(url: str) -> str:
    return url.split('/')[2]


def make_web():
    """A small site: news.yahoo.com pages link to each other and to other hosts."""
    h = 'http://news.yahoo.com'
    links = {
        f'{h}/': [f'{h}/world', f'{h}/sport', f'{h}/tech', 'http://news.google.com/'],
        f'{h}/world': [f'{h}/world/asia', f'{h}/world/europe', f'{h}/'],
        f'{h}/sport': [f'{h}/sport/f1', f'{h}/sport/tennis', 'http://yahoo.com/sport'],
        f'{h}/tech': [f'{h}/tech/ai', f'{h}/tech/chips', f'{h}/world'],
        f'{h}/world/asia': [f'{h}/world/asia/japan', f'{h}/world/asia/taiwan'],
        f'{h}/world/europe': [f'{h}/world/europe/france'],
        f'{h}/sport/f1': [f'{h}/sport/f1/monaco'],
        f'{h}/sport/tennis': [],
        f'{h}/tech/ai': [f'{h}/tech/ai/llm', 'http://news.yahoo.com.evil.example/ai'],
        f'{h}/tech/chips': [f'{h}/tech/ai'],
        f'{h}/world/asia/japan': [],
        f'{h}/world/asia/taiwan': [f'{h}/'],
        f'{h}/world/europe/france': [],
        f'{h}/sport/f1/monaco': [],
        f'{h}/tech/ai/llm': [f'{h}/tech/ai/llm/agents'],
        f'{h}/tech/ai/llm/agents': [],
        'http://news.google.com/': ['http://news.google.com/a'],
        'http://yahoo.com/sport': [],
        'http://news.yahoo.com.evil.example/ai': [],
    }
    return links


class HtmlParser:
    """LeetCode's interface; each call costs `latency` seconds, like a real fetch."""

    def __init__(self, links, latency=0.05):
        self.links, self.latency, self.calls = links, latency, 0
        self._lock = threading.Lock()

    def getUrls(self, url):
        with self._lock:
            self.calls += 1
        time.sleep(self.latency)
        return list(self.links.get(url, []))


def crawl_serial(startUrl, parser):
    host, seen, stack = hostname(startUrl), {startUrl}, [startUrl]
    while stack:
        for v in parser.getUrls(stack.pop()):
            if hostname(v) == host and v not in seen:
                seen.add(v)
                stack.append(v)
    return seen


def crawl_substring_bug(startUrl, parser):
    """Checks `host in url` instead of comparing host names exactly."""
    host, seen, stack = hostname(startUrl), {startUrl}, [startUrl]
    while stack:
        for v in parser.getUrls(stack.pop()):
            if host in v and v not in seen:
                seen.add(v)
                stack.append(v)
    return seen


class Solution1242:
    """Breadth-first by waves: every page of the current wave is fetched at once."""

    def crawl(self, startUrl, htmlParser, workers=16):
        hostname = lambda url: url.split('/')[2]     # self-contained, as LeetCode needs
        host, seen, frontier, waves = hostname(startUrl), {startUrl}, [startUrl], 0
        with ThreadPoolExecutor(workers) as pool:
            while frontier:
                waves += 1
                nxt = []
                for urls in pool.map(htmlParser.getUrls, frontier):
                    for v in urls:
                        if hostname(v) == host and v not in seen:
                            seen.add(v)
                            nxt.append(v)
                frontier = nxt
        self.waves = waves
        return list(seen)


def bfs_depth(links, start):
    """Longest shortest-path (in links) from start inside the start host + 1 = waves."""
    host, dist, q = hostname(start), {start: 1}, [start]
    for u in q:
        for v in links.get(u, []):
            if hostname(v) == host and v not in dist:
                dist[v] = dist[u] + 1
                q.append(v)
    return max(dist.values())


web, start = make_web(), 'http://news.yahoo.com/'
p = HtmlParser(web); t0 = time.perf_counter(); got = crawl_serial(start, p)
print(f'serial : {len(got)} pages, {p.calls} fetches, {time.perf_counter() - t0:.2f} s')
p = HtmlParser(web); s = Solution1242(); t0 = time.perf_counter(); got2 = s.crawl(start, p)
print(f'threads: {len(got2)} pages, {s.waves} waves, {time.perf_counter() - t0:.2f} s')
print('`in` instead of ==:', sorted(set(crawl_substring_bug(start, HtmlParser(web, 0))) - got))

serial : 16 pages, 16 fetches, 0.80 s
threads: 16 pages, 5 waves, 0.26 s
`in` instead of ==: ['http://news.yahoo.com.evil.example/ai']


## 8. Work and depth

Draw a computation as a graph of operations. **Work** W is the number of operations;
**depth** D is the longest chain in which each needs the previous one. With p workers a
greedy scheduler finishes in T_p steps with `max(W/p, D) <= T_p <= W/p + D` (Brent).
A loop has D = W: no number of workers helps. A tree sum has D = log2(n). Hillis-Steele's
scan has the smallest depth but nine times the work, so on 4 workers it is slower than a
plain loop; Blelloch's scan keeps the work at about 2n.

In [9]:
# ---------------------------------------------------------------- 8. work and depth
class Dag:
    """A computation as a graph: every node is one addition, its inputs are earlier
    nodes or input values.  work = number of additions, depth = longest chain."""

    def __init__(self, n_inputs):
        self.n_inputs = n_inputs
        self.deps = []                      # deps[k] for addition node k
        self.level = []                     # longest chain ending at node k

    def inp(self, i):
        return ('in', i)

    def add(self, a, b):
        lv = 1 + max(self.level[x[1]] if x[0] == 'op' else 0 for x in (a, b))
        self.deps.append((a, b))
        self.level.append(lv)
        return ('op', len(self.deps) - 1)

    @property
    def work(self):
        return len(self.deps)

    @property
    def depth(self):
        return max(self.level, default=0)

    def evaluate(self, values, outputs):
        val = []
        get = lambda x: values[x[1]] if x[0] == 'in' else val[x[1]]
        for a, b in self.deps:
            val.append(get(a) + get(b))
        return [get(o) if o is not None else 0 for o in outputs]

    def schedule(self, p):
        """Greedy: at every tick run up to p additions whose inputs are ready
        (lowest node id first).  Returns the list of ticks, each a list of node ids."""
        import heapq
        waiting = [sum(x[0] == 'op' for x in d) for d in self.deps]
        users = defaultdict(list)
        for k, d in enumerate(self.deps):
            for x in d:
                if x[0] == 'op':
                    users[x[1]].append(k)
        ready = [k for k in range(self.work) if waiting[k] == 0]
        heapq.heapify(ready)
        ticks = []
        while ready:
            run = [heapq.heappop(ready) for _ in range(min(p, len(ready)))]
            ticks.append(run)
            for k in run:
                for u in users[k]:
                    waiting[u] -= 1
                    if waiting[u] == 0:
                        heapq.heappush(ready, u)
        return ticks


def sum_chain(n):
    g = Dag(n)
    acc = g.inp(0)
    for i in range(1, n):
        acc = g.add(acc, g.inp(i))
    return g, [acc]


def sum_tree(n):
    g = Dag(n)
    level = [g.inp(i) for i in range(n)]
    while len(level) > 1:
        nxt = [g.add(level[i], level[i + 1]) for i in range(0, len(level) - 1, 2)]
        if len(level) % 2:
            nxt.append(level[-1])
        level = nxt
    return g, level


def scan_serial(n):
    g = Dag(n)
    out = [g.inp(0)]
    for i in range(1, n):
        out.append(g.add(out[-1], g.inp(i)))
    return g, out


def scan_hillis_steele(n):
    """Inclusive scan: at round d every position i >= 2^d adds the value 2^d back."""
    g = Dag(n)
    cur = [g.inp(i) for i in range(n)]
    d = 1
    while d < n:
        cur = [cur[i] if i < d else g.add(cur[i - d], cur[i]) for i in range(n)]
        d *= 2
    return g, cur


def scan_blelloch(n):
    """Exclusive scan (n a power of two): an up-sweep that builds partial sums in a
    tree, then a down-sweep that hands each subtree the sum of everything left of it."""
    g = Dag(n)
    cur = [g.inp(i) for i in range(n)]
    d = 1
    while d < n:
        for i in range(2 * d - 1, n, 2 * d):
            cur[i] = g.add(cur[i - d], cur[i])
        d *= 2
    cur[n - 1] = None                               # the identity, 0
    d = n // 2
    while d >= 1:
        for i in range(2 * d - 1, n, 2 * d):
            left = cur[i - d]
            cur[i - d] = cur[i]
            cur[i] = left if cur[i] is None else g.add(left, cur[i])
        d //= 2
    return g, cur


def brent_bounds(work, depth, p):
    return max(math.ceil(work / p), depth), math.ceil(work / p) + depth


def amdahl(serial_fraction, p):
    return 1 / (serial_fraction + (1 - serial_fraction) / p)


for name, b in (('sum loop', sum_chain), ('sum tree', sum_tree), ('scan loop', scan_serial),
                ('Hillis-Steele', scan_hillis_steele), ('Blelloch', scan_blelloch)):
    g, _ = b(1024)
    print(f'{name:<14} W={g.work:<5} D={g.depth:<5}',
          ' '.join(f'T_{p}={len(g.schedule(p))}' for p in (1, 4, 16, 64, 256, 1024)))
print('Amdahl, 5% serial:', [round(amdahl(0.05, p), 1) for p in (2, 4, 8, 16, 64)])

sum loop       W=1023  D=1023  T_1=1023 T_4=1023 T_16=1023 T_64=1023 T_256=1023 T_1024=1023
sum tree       W=1023  D=10    T_1=1023 T_4=257 T_16=67 T_64=21 T_256=11 T_1024=10
scan loop      W=1023  D=1023  T_1=1023 T_4=1023 T_16=1023 T_64=1023 T_256=1023 T_1024=1023
Hillis-Steele  W=9217  D=10    T_1=9217 T_4=2305 T_16=577 T_64=145 T_256=37 T_1024=10
Blelloch       W=2036  D=18    T_1=2036 T_4=509 T_16=128 T_64=34 T_256=19 T_1024=18
Amdahl, 5% serial: [1.9, 3.5, 5.9, 9.1, 15.4]


## 9. Map-reduce

Map each document to word counts, shuffle every word to reducer `bucket(word)`, reduce
by summing. Two silent failures: every mapper must put a word in the *same* bucket
(Python's `hash()` of a string differs between processes started with `spawn`, the
default on Windows and macOS), and the reduce step must combine pieces whose sizes differ
(a mean of means is not the mean).

In [10]:
# ---------------------------------------------------------------- 9. map-reduce
VOCAB = ('the of and to in a is that for it as was with be by on not he this are '
         'or his from at which but have an they you were her she there one all we '
         'their been has would when who will more if no out so said what up its '
         'about into than them can only other new some could time these two may').split()


def make_corpus(docs=40, words=400, seed=43):
    rng = random.Random(seed)
    weights = [1 / (r + 1) for r in range(len(VOCAB))]       # Zipf: a few words dominate
    return [rng.choices(VOCAB, weights, k=words) for _ in range(docs)]


def stable_bucket(word, reducers, salt=b''):
    """Same word -> same reducer, in every process (blake2b, not hash())."""
    h = hashlib.blake2b(word.encode(), digest_size=8, salt=salt.ljust(16, b'\0')[:16])
    return int.from_bytes(h.digest(), 'big') % reducers


def map_doc(doc, reducers=4, salt=b''):
    """Map + combine: count inside the document, then split the counts by reducer."""
    counts = Counter(doc)
    parts = [Counter() for _ in range(reducers)]
    for w, c in counts.items():
        parts[stable_bucket(w, reducers, salt)][w] += c
    return parts


def map_doc_builtin_hash(doc, reducers=4):
    """The same mapper with Python's hash(): its value is randomised per process."""
    parts = [Counter() for _ in range(reducers)]
    for w, c in Counter(doc).items():
        parts[hash(w) % reducers][w] += c
    return parts


def reduce_parts(all_parts, reducers=4):
    """Shuffle (reducer r gets part r of every mapper) + reduce (sum), then write the
    reducers' outputs one after another into one table, as a real job's output files."""
    out = {}
    for r in range(reducers):
        total = Counter()
        for parts in all_parts:
            total.update(parts[r])
        out.update(total)                # a word seen by two reducers: last one wins
    return out


def mapreduce(docs, reducers=4, workers=2, salts=None, ctx=None, mapper=None):
    if salts is not None:                # simulate mappers that disagree on the bucket
        all_parts = [map_doc(d, reducers, salts[i % len(salts)]) for i, d in enumerate(docs)]
    else:
        with (ctx or mp).Pool(workers) as pool:
            all_parts = pool.starmap(mapper or map_doc, [(d, reducers) for d in docs])
    return reduce_parts(all_parts, reducers)


def mean_of_means(chunks):
    return sum(sum(c) / len(c) for c in chunks) / len(chunks)


def mean_by_sum_count(chunks):
    s, n = reduce(lambda a, b: (a[0] + b[0], a[1] + b[1]), ((sum(c), len(c)) for c in chunks))
    return s / n


def tree_reduce(xs, op):
    while len(xs) > 1:
        xs = [op(xs[i], xs[i + 1]) if i + 1 < len(xs) else xs[i] for i in range(0, len(xs), 2)]
    return xs[0]


docs = make_corpus()
truth = Counter(w for d in docs for w in d)
print(len(docs), 'documents,', sum(truth.values()), 'words, top 3', truth.most_common(3))
print('2 processes, 4 reducers == Counter:', mapreduce(docs, mapper=concurrency.map_doc) == dict(truth))
sim = mapreduce(docs, salts=[b'mapper-0', b'mapper-1'])
print('mappers that disagree on the bucket:', sum(sim[w] != truth[w] for w in truth), 'words wrong')
spawn = mapreduce(docs, ctx=mp.get_context('spawn'), mapper=concurrency.map_doc_builtin_hash)
print('hash() + spawn workers:', sum(spawn.get(w) != truth[w] for w in truth), 'words wrong (this run)')
chunks = [[120, 80, 100], [400], [90, 110, 100, 95, 105, 100]]
print('mean of means', mean_of_means(chunks), ' true mean', mean_by_sum_count(chunks))
print('ten 0.1s: loop', sum([0.1] * 10), ' tree', tree_reduce([0.1] * 10, lambda a, b: a + b))

40 documents, 16000 words, top 3 [('the', 3371), ('of', 1640), ('and', 1129)]
2 processes, 4 reducers == Counter: True
mappers that disagree on the bucket: 51 words wrong
hash() + spawn workers: 48 words wrong (this run)
mean of means 200.0  true mean 130.0
ten 0.1s: loop 0.9999999999999999  tree 1.0


## Tests

In [11]:
def check() -> None:
    # exact model-checker counts
    c2 = xcount(count_schedules([incrementer()] * 2, {'x': 0}))
    assert c2 == Counter({1: 4, 2: 2})
    c3 = xcount(count_schedules([incrementer()] * 3, {'x': 0}))
    assert c3 == Counter({1: 48, 2: 36, 3: 6})
    assert xcount(count_schedules([incrementer(locked=True)] * 3, {'x': 0})) == Counter({3: 6})
    assert count_schedules(two_lock_threads(False), {})[DEADLOCK] == 2
    assert count_schedules(two_lock_threads(True), {})[DEADLOCK] == 0
    assert reachable_states(philosophers(5), {}) == (9302, 1)
    assert reachable_states(philosophers(5, ordered=True), {})[1] == 0
    c = count_schedules(philosophers(5), {})
    assert c[DEADLOCK] == 120 and sum(c.values()) == 2_430_842_120
    assert len(count_schedules(print_in_order_model(False), {})) == 6
    assert [o[1] for o in count_schedules(print_in_order_model(True), {})] == ['fst']
    # real threads where the answer must not depend on timing
    assert hammer(bump_with_call, lock=threading.Lock()) == 800_000
    assert set(lc1114_all_orders()) == {'firstsecondthird'}
    meals, log = lc1226_run()
    assert meals == 15 and forks_never_shared(log)
    assert real_deadlock() == list(range(5)) and real_deadlock(ordered=True) == []
    # crawler
    web, start = make_web(), 'http://news.yahoo.com/'
    want = crawl_serial(start, HtmlParser(web, 0))
    assert len(want) == 16 and all(hostname(u) == 'news.yahoo.com' for u in want)
    sol = Solution1242()
    assert set(sol.crawl(start, HtmlParser(web, 0))) == want
    assert sol.waves == bfs_depth(web, start) == 5
    assert len(crawl_substring_bug(start, HtmlParser(web, 0))) == 17
    # work / depth: the DAGs compute the right answers
    from itertools import accumulate as acc
    for n in (8, 16, 64):
        vals = list(range(3, 3 + n))
        assert sum_chain(n)[0].evaluate(vals, sum_chain(n)[1]) == [sum(vals)]
        g, o = sum_tree(n)
        assert g.evaluate(vals, o) == [sum(vals)]
        for b in (scan_serial, scan_hillis_steele):
            g, o = b(n)
            assert g.evaluate(vals, o) == list(acc(vals))
        g, o = scan_blelloch(n)
        assert g.evaluate(vals, o) == [0] + list(acc(vals))[:-1]
    for b in (sum_chain, sum_tree, scan_serial, scan_hillis_steele, scan_blelloch):
        g, _ = b(1024)
        for p in (1, 4, 64, 1024):
            lo, hi = brent_bounds(g.work, g.depth, p)
            assert lo <= len(g.schedule(p)) <= hi
    assert (sum_tree(1024)[0].work, sum_tree(1024)[0].depth) == (1023, 10)
    assert (scan_hillis_steele(1024)[0].work, scan_blelloch(1024)[0].work) == (9217, 2036)
    # map-reduce
    docs = make_corpus()
    truth = Counter(w for d in docs for w in d)
    assert reduce_parts([map_doc(d) for d in docs]) == dict(truth)
    assert mean_by_sum_count([[1, 2], [3]]) == 2 and mean_of_means([[1, 2], [3]]) == 2.25
    assert tree_reduce([0.1] * 10, lambda a, b: a + b) != sum([0.1] * 10)


check()
print('all assertions passed')

all assertions passed
